# Car Parking Vision — PKLot CNN Capstone

This notebook builds a binary classifier for an **individual parking-space crop**: Empty (0) vs Occupied (1). It downloads PKLot, prepares a reproducible split, measures a majority-class baseline, trains a CNN from scratch, and provides a final held-out test evaluation.

**Protocol:** UFPR04 + UFPR05 are split by capture date into train/validation; PUCPR is reserved as a separate-site test set. Do not use test metrics to tune the model.

Run cells in order. Dataset files, crops, checkpoints, and outputs stay in the temporary Colab runtime and are not committed to Git.

In [ ]:
from pathlib import Path
import subprocess, sys, os

REPO_URL = "https://github.com/173eiei/fundl-project.git"
PROJECT_DIR = Path("/content/fundl-project")
if not (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
print("Project:", PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
import torch, pandas as pd, matplotlib.pyplot as plt
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 1. Runtime and reproducibility

Select **Runtime → Change runtime type → T4 GPU** if available. Seed 42 is used by preprocessing and training.

In [ ]:
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Training can run on CPU, but will be much slower.")
print("Device:", DEVICE, "| Seed:", SEED)

## 2. Download PKLot

The original archive is approximately 4.6 GB. It is stored under /content, not in the Git repository. The official UFPR URL is tried first; a Hugging Face mirror fallback is provided below.

In [ ]:
ARCHIVE = Path("/content/PKLot.tar.gz")
OFFICIAL_URL = "https://www.inf.ufpr.br/vri/databases/PKLot.tar.gz"
if not ARCHIVE.exists() or ARCHIVE.stat().st_size == 0:
    result = subprocess.run(["wget", "-c", "-O", str(ARCHIVE), OFFICIAL_URL])
    if result.returncode != 0 or not ARCHIVE.exists() or ARCHIVE.stat().st_size == 0:
        raise RuntimeError("Official download failed. Run the Hugging Face mirror fallback cell below.")
print(f"Archive size: {ARCHIVE.stat().st_size / (1024**3):.2f} GiB")

In [ ]:
# Fallback only if the official UFPR host cannot be reached.
# The mirror describes its PKLot.tar.gz as byte-for-byte identical to the original archive.
if not ARCHIVE.exists() or ARCHIVE.stat().st_size == 0:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
    from huggingface_hub import hf_hub_download
    downloaded = hf_hub_download(
        repo_id="teenygrad/pklot", filename="PKLot.tar.gz",
        repo_type="dataset", local_dir="/content/pklot_mirror",
    )
    ARCHIVE = Path(downloaded)
    print("Using mirror archive:", ARCHIVE)
else:
    print("Official archive already downloaded; fallback not needed.")

## 3. Extract and inspect the archive

The preparation script supports the segmented crop folders when present. If only full-scene images and XML annotations are present, it generates perspective-corrected parking-space crops from the four-point XML contours.

In [ ]:
DATA_ROOT = Path("/content/pklot")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
if not any(DATA_ROOT.iterdir()):
    subprocess.run(["tar", "-xzf", str(ARCHIVE), "-C", str(DATA_ROOT)], check=True)
segmented_dirs = [p for p in DATA_ROOT.rglob("*") if p.is_dir() and p.name.lower() == "pklotsegmented"]
xml_count = sum(1 for _ in DATA_ROOT.rglob("*.xml"))
print("Dataset root:", DATA_ROOT)
print("PKLotSegmented directories:", len(segmented_dirs))
print("XML annotations:", xml_count)
if not segmented_dirs and xml_count == 0:
    raise RuntimeError("Could not find segmented crops or PKLot XML annotations after extraction.")

## 4. Prepare crops and create the split manifest

The script assigns **PUCPR entirely to test**. Dates from UFPR04/UFPR05 are grouped before the fixed-seed train/validation split, so one capture date cannot appear in both train and validation.

In [ ]:
subprocess.run([
    sys.executable, "-m", "src.prepare_data",
    "--dataset-root", str(DATA_ROOT),
    "--output-dir", "data/processed",
    "--seed", str(SEED),
    "--val-fraction", "0.20",
], check=True)
MANIFEST = Path("data/processed/manifest.csv")
manifest = pd.read_csv(MANIFEST)
display(manifest.groupby(["split", "site", "class_name"]).size().rename("samples").reset_index())
print("Unique dates by split:")
display(manifest.groupby("split")["date"].nunique())

## 5. Inspect examples and class balance

Check that crops and folder labels look sensible before training. These examples are sampled from the training split only.

In [ ]:
from PIL import Image

train_rows = manifest[manifest["split"] == "train"]
sample_rows = train_rows.sample(n=min(8, len(train_rows)), random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    with Image.open(row["path"]) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(f"{row['class_name']} | {row['site']} | {row['weather']}")
    ax.axis("off")
for ax in axes.flat[len(sample_rows):]:
    ax.axis("off")
plt.tight_layout()
plt.show()
display(manifest.groupby(["split", "class_name"]).size().unstack(fill_value=0))

## 6. Majority-class baseline

The majority class is learned from **training labels only**. This cell reports its performance on validation data; it does not use the test set.

In [ ]:
subprocess.run([sys.executable, "-m", "src.baseline", "--manifest", str(MANIFEST)], check=True)
baseline = pd.read_json("outputs/results/baseline_validation.json", typ="series")
display(baseline.to_frame("value"))

## 7. Train the CNN from scratch

Architecture: four convolutional blocks, batch normalization, max pooling, dropout, global average pooling, and a two-class classifier. Training uses weighted cross-entropy (weights from train labels only), AdamW, augmentation, weight decay, a validation-loss scheduler, and early stopping.

In [ ]:
subprocess.run([
    sys.executable, "-m", "src.train",
    "--manifest", str(MANIFEST),
    "--epochs", "20",
    "--batch-size", "64",
    "--image-size", "128",
    "--learning-rate", "0.001",
    "--weight-decay", "0.0001",
    "--patience", "5",
    "--seed", str(SEED),
], check=True)

## 8. Review learning curves

Use train/validation curves to discuss overfitting and justify regularization. Hyperparameter choices must be made using validation data only.

In [ ]:
history = pd.read_csv("outputs/results/history.csv")
display(history.tail())
from IPython.display import display, Image as DisplayImage
display(DisplayImage(filename="outputs/results/learning_curves.png"))

## 9. Final held-out test evaluation — run once after choices are locked

**Do not use this result to tune the model.** The test split is the separate PUCPR site. This cell reports accuracy, precision, recall, F1, a majority-baseline comparison, a confusion matrix, and saved misclassified examples.

In [ ]:
subprocess.run([
    sys.executable, "-m", "src.evaluate",
    "--manifest", str(MANIFEST),
    "--checkpoint", "outputs/models/best.pt",
], check=True)
test_metrics = pd.read_json("outputs/results/test_metrics.json", typ="series")
display(test_metrics)

## 10. Single-crop inference demo

Upload one **already-cropped parking-space image** (not a full parking-lot scene). The model returns Empty/Occupied probabilities.

In [ ]:
from google.colab import files
uploaded = files.upload()
if uploaded:
    image_name = next(iter(uploaded))
    subprocess.run([
        sys.executable, "-m", "src.predict",
        "--image", image_name,
        "--checkpoint", "outputs/models/best.pt",
    ], check=True)

## Reproducibility notes for the report

Record the Colab GPU model, actual split counts, class balance, hyperparameters, best epoch, learning curves, final test metrics, and at least one correctly predicted and one incorrectly predicted PUCPR crop. Do not claim a score until the test cell has genuinely run. Dataset, crops, checkpoints, and outputs are ignored by Git.